# Notebook 08 — New Bitcoin Twitter Dataset: Engagement-Weighted Daily Features
**Dataset:** Gigasheet Bitcoin Tweets Sentiment Analysis (2022-01-01 → 2023-06-22)

**Columns used:** date, text, sentiment_label, sentiment_score, reply_count, like_count, retweet_count, quote_count

**Output:** `bitcoin_twitter_daily_new.csv` — daily aggregated features for merge into training pipeline

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Config ─────────────────────────────────────────────────────────────────
TWITTER_CSV   = "bitcoin_tweets_sentiment.csv"   # rename your Gigasheet download to this
OUTPUT_CSV    = "bitcoin_twitter_daily_new.csv"
DATE_MIN      = pd.Timestamp("2022-01-01")
DATE_MAX      = pd.Timestamp("2023-06-22")
LOG_CAP       = 10.0   # cap for log-engagement weight (prevents viral tweet dominance)

print(f"Config: restrict to {DATE_MIN.date()} → {DATE_MAX.date()}")

In [ ]:
# ── Load ────────────────────────────────────────────────────────────────────
df = pd.read_csv(TWITTER_CSV, low_memory=False)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

In [ ]:
# ── Date parsing & filter ───────────────────────────────────────────────────
df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True).dt.tz_localize(None).dt.normalize()
df = df.dropna(subset=["date"])
df = df[(df["date"] >= DATE_MIN) & (df["date"] <= DATE_MAX)].copy()
print(f"After date filter: {len(df):,} tweets  |  {df['date'].nunique()} unique days")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

In [ ]:
# ── Engagement weight ───────────────────────────────────────────────────────
# Raw engagement columns — coerce to numeric, fill NaN → 0
eng_cols = ["reply_count", "like_count", "retweet_count", "quote_count"]
for c in eng_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).clip(lower=0)

df["raw_engagement"] = df["reply_count"] + df["like_count"] + df["retweet_count"] + df["quote_count"]

# Log-capped weight:  w = min(log1p(engagement), LOG_CAP)
# Tweets with 0 engagement → w = log1p(0) = 0 → then we floor at 0.01 so they still contribute weakly
df["eng_weight"] = np.log1p(df["raw_engagement"]).clip(upper=LOG_CAP)
df["eng_weight"] = df["eng_weight"].where(df["raw_engagement"] > 0, other=0.01)   # weak floor for zero-engagement tweets

print("Engagement weight distribution:")
print(df["eng_weight"].describe().round(4))

In [ ]:
# ── Sentiment numeric ───────────────────────────────────────────────────────
df["sentiment_score"] = pd.to_numeric(df["sentiment_score"], errors="coerce").fillna(0)
df["is_positive"] = (df["sentiment_label"].str.lower().str.strip() == "positive").astype(float)
df["is_negative"] = (df["sentiment_label"].str.lower().str.strip() == "negative").astype(float)

# Weighted sentiment
df["w_sentiment"] = df["sentiment_score"] * df["eng_weight"]
df["w_positive"]  = df["is_positive"]     * df["eng_weight"]
df["w_negative"]  = df["is_negative"]     * df["eng_weight"]
df["w_sq_sentiment"] = (df["sentiment_score"] ** 2) * df["eng_weight"]  # for weighted std

print("Sentiment label distribution:")
print(df["sentiment_label"].value_counts())

In [ ]:
# ── Daily aggregation ───────────────────────────────────────────────────────
g = df.groupby("date")

daily = pd.DataFrame(index=g.groups.keys())
daily.index.name = "date"

w_sum = g["eng_weight"].sum().rename("_w_sum")

daily["weighted_sentiment_mean"] = (g["w_sentiment"].sum() / w_sum).round(6)
daily["weighted_positive_share"] = (g["w_positive"].sum()  / w_sum).round(6)
daily["weighted_negative_share"] = (g["w_negative"].sum()  / w_sum).round(6)

# Weighted variance → weighted std
mean_sq = (g["w_sq_sentiment"].sum() / w_sum)
sq_mean = daily["weighted_sentiment_mean"] ** 2
daily["weighted_sentiment_std"] = np.sqrt((mean_sq - sq_mean).clip(lower=0)).round(6)

daily["weighted_tweet_count"]  = w_sum.round(4)          # total engagement-weighted "tweet mass"
daily["total_engagement"]      = g["raw_engagement"].sum().round(0)
daily["average_engagement"]    = g["raw_engagement"].mean().round(4)
daily["max_engagement"]        = g["raw_engagement"].max().round(0)
daily["raw_tweet_count"]       = g["raw_engagement"].count()

daily = daily.reset_index().sort_values("date")
daily["coin"] = "Bitcoin"

# Rolling features (3-day, 7-day)
for w in [3, 7]:
    daily[f"sent_ma{w}"] = daily["weighted_sentiment_mean"].rolling(w, min_periods=1).mean().round(6)
    daily[f"sent_chg{w}"] = daily["weighted_sentiment_mean"].pct_change(w).round(6)
    daily[f"eng_ma{w}"]  = daily["average_engagement"].rolling(w, min_periods=1).mean().round(4)

print(f"Daily rows: {len(daily)}")
print(f"Date range: {daily['date'].min().date()} → {daily['date'].max().date()}")
daily.head()

In [ ]:
# ── Save ────────────────────────────────────────────────────────────────────
daily.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved {OUTPUT_CSV}  shape={daily.shape}")
print("Columns:", daily.columns.tolist())

# Also save date range metadata
twitter_meta = {
    "date_min": str(daily["date"].min().date()),
    "date_max": str(daily["date"].max().date()),
    "n_days": len(daily),
    "source": "Gigasheet Bitcoin Tweets Sentiment Analysis",
    "engagement_weight": "log1p capped at 10, floor 0.01 for zero-engagement",
    "feature_cols": [c for c in daily.columns if c not in ["date", "coin"]]
}
with open("twitter_metadata.json", "w") as f:
    json.dump(twitter_meta, f, indent=2)
print("✅ Saved twitter_metadata.json")
print(json.dumps(twitter_meta, indent=2))